In [1]:
import os
os.chdir("..")

In [2]:
import asyncio
from datetime import datetime, timedelta
from typing import Optional
from pprint import pprint

# Jupyter + asyncio
import nest_asyncio
nest_asyncio.apply()

In [12]:
from sqlalchemy.ext.asyncio import AsyncSession
from sqlmodel import select
from src.database.connection import async_session
from src.database.schema import Article, Event, Weather
from src.ai.summary_generator import SummaryGenerator


In [26]:
target_date: Optional[datetime] = None
if target_date is None:
            target_date = (
                datetime.utcnow()
                .replace(hour=0, minute=0, second=0, microsecond=0)- timedelta(days=1)
            )
date_start = target_date
date_end = date_start + timedelta(days=1)

date_start,date_end

(datetime.datetime(2026, 1, 12, 0, 0), datetime.datetime(2026, 1, 13, 0, 0))

In [30]:
async def inspect_summary_input(
    target_date: Optional[datetime] = None
):
    generator = SummaryGenerator()

    async with async_session() as session:
        # --- dokładnie to samo co w generate_daily_summary ---
        if target_date is None:
            target_date = (
                datetime.utcnow()
                .replace(hour=0, minute=0, second=0, microsecond=0)- timedelta(days=1)
            )

        date_start = target_date
        date_end = date_start + timedelta(days=1)

        # Artykuły
        articles_result = await session.execute(
            select(Article)
            .where(Article.published_at >= date_start)
            .where(Article.published_at <= date_end)
            .where(Article.processed == True)
            .order_by(Article.published_at.desc())
        )
        articles = articles_result.scalars().all()

        articles_by_category = {}
        for article in articles:
            category = article.category or "Inne"
            articles_by_category.setdefault(category, []).append(article)

        # Wydarzenia
        events_result = await session.execute(
            select(Event)
            .where(Event.event_date >= date_end)
            .where(Event.event_date < date_end + timedelta(days=7))
            .order_by(Event.event_date.asc())
            .limit(10)
        )
        events = events_result.scalars().all()

        # Pogoda
        weather_result = await session.execute(
            select(Weather)
            .where(Weather.is_current == True)
            .order_by(Weather.fetched_at.desc())
            .limit(1)
        )
        weather = weather_result.scalar_one_or_none()

        # INPUT DLA AI (TO NAS INTERESUJE)
        input_text = generator._prepare_input_for_ai(
            date_start,
            articles_by_category,
            events,
            weather
        )

        return {
            "date": date_start,
            "articles_count": len(articles),
            "categories": list(articles_by_category.keys()),
            "events_count": len(events),
            "weather": weather,
            "input_text": input_text
        }


In [31]:
data = await inspect_summary_input()

print("DATA:", data["date"])
print("LICZBA ARTYKUŁÓW:", data["articles_count"])
print("KATEGORIE:", data["categories"])
print("LICZBA WYDARZEŃ:", data["events_count"])


DATA: 2026-01-12 00:00:00
LICZBA ARTYKUŁÓW: 7
KATEGORIE: ['Kultura', 'Rekreacja', 'Urząd']
LICZBA WYDARZEŃ: 10


In [29]:
print(data["input_text"])


Data: 2026-01-12
Liczba artykułów: 7

ARTYKUŁY PO KATEGORIACH:


## KULTURA (5 artykułów):

1. 📣 UWAGA RODZICE, TRENERZY I OPIEKUNOWIE SPORTOWCÓW!

Rozpoczynamy nabór wniosków o wyróżnienia sport...
   → Rozpoczęto nabór wniosków o wyróżnienia sportowe w ramach Plebiscytu Sportowego organizowanego przez Serwis Informacyjny Syla. Inicjatywa dotyczy młodych sportowców z Gminy Rybno, a finał plebiscytu odbędzie się 13 lutego w Świetlicy Wiejskiej w Hartowcu.
   📍 Rybno

2. 🎭✨ Bal Karnawałowy w Truszczynach – zapraszamy! ✨🎭

Karnawał to idealny czas na zabawę, taniec i wsp...
   → Zaplanowano Bal Karnawałowy w Truszczynach, który odbędzie się 7 lutego 2026 roku w remizie OSP. W programie przewidziano muzykę na żywo, bogate menu oraz świetną atmosferę. Zapisy trwają do 28 stycznia.
   📍 Truszczyny

3. RyBaśka - Restauracja Rybna wraca do stałych godzin otwarcia 🐟🐠

⤵⤵⤵
   → Restauracja RyBaśka w Rybnie wraca do stałych godzin otwarcia. Informacja ta ma znaczenie dla mieszkańców i gości, któ